In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


In [ ]:
root_dir = Path().resolve().parents[0]
sunset_path = root_dir / "data" / "sunset_data.csv"


provider_release_paths = {
    "GPT": root_dir / "data" / "chatgpt_model_updates.csv",
    "Claude": root_dir / "data" / "claude_model_updates.csv",
    "Gemini": root_dir / "data" / "gemini_model_updates.csv",
    "Grok": root_dir / "data" / "grok_model_updates.csv",
}

sunset_df = pd.read_csv(sunset_path)
sunset_df.head()


In [ ]:
# Merge related departments into broader groups so we stay within max 10 plots.
MERGED_GROUPS = {
    "Computing & EECS": {"CSE", "DSC", "ECE"},
    "Engineering": {"MAE", "BENG", "NANO"},
    "Physical Sciences": {"CHEM", "PHYS", "SIO"},
    "Life Sciences": {"BILD", "BICD", "BIEB", "BIMM", "BIBC", "BIPN"},
    "Math & Logic": {"MATH", "LIGN"},
    "Econ & Management": {"ECON", "MGT"},
    "Social Sciences": {"POLI", "PSYC", "SOCI", "USP", "COGS", "HDS"},
    "Writing & Humanities": {"CAT", "DOC", "WCWP", "MCWP", "HUM", "MMW", "HILD", "ETHN", "PHIL"},
    "Arts & Media": {"MUS", "VIS", "TDGE", "TDMV"},
    "Languages": {"JAPN", "CHIN", "LTEN", "LIGM", "LIFR", "LISP", "LTWL", "LTKO", "LTSP", "LTEA", "LTWR"},
}

DEPT_TO_GROUP = {dept: group for group, depts in MERGED_GROUPS.items() for dept in depts}

# All four provider milestone dates used for comparison overlays.
LLM_MARKERS = {
    "GPT": pd.Timestamp("2022-11-30"),      # GPT-3.5
    "Claude": pd.Timestamp("2023-03-14"),   # Claude 1
    "Gemini": pd.Timestamp("2023-12-06"),   # Gemini 1.0
    "Grok": pd.Timestamp("2023-11-04"),     # Grok-1
}

BAND_TO_GRADES = {
    "A": ["A+", "A", "A-"],
    "B": ["B+", "B", "B-"],
    "C": ["C+", "C", "C-"],
    "D/F": ["D", "F"],
}

MIN_QUARTERS = 12  # 4 years of quarter data
MAX_GROUPS = 10
QUALITY_RATIO_RANGE = (0.95, 1.05)

COVID_START = pd.Timestamp("2020-03-01")
COVID_REMOTE_END = pd.Timestamp("2021-09-01")


In [ ]:
def extract_department(course):
    if pd.isna(course):
        return None
    match = re.match(r"^\s*([A-Za-z&]+)", str(course))
    return match.group(1).upper() if match else None


def parse_grade_distribution(raw):
    if pd.isna(raw):
        return {}

    text = str(raw).strip()
    lowered = text.lower()
    if not text or "not available" in lowered or "temporarily unavailable" in lowered:
        return {}

    parsed = {}
    for piece in text.split(","):
        if ":" not in piece:
            continue
        key, value = piece.split(":", 1)
        key = key.strip()
        value = value.strip()

        if key == "Class GPA":
            continue

        try:
            parsed[key] = parsed.get(key, 0) + int(value)
        except ValueError:
            continue

    return parsed


def term_to_date(term):
    match = re.match(r"^(Winter|Spring|Fall)\s+Qtr\s+(\d{4})$", str(term))
    if not match:
        return pd.NaT

    quarter_to_month = {"Winter": 1, "Spring": 4, "Fall": 9}
    quarter, year = match.group(1), int(match.group(2))
    return pd.Timestamp(year=year, month=quarter_to_month[quarter], day=1)


def prepare_group_term_data(df):
    work = df[["Term", "Course", "Professor", "Grade distribution"]].copy()
    work["Department"] = work["Course"].apply(extract_department)
    work["Group"] = work["Department"].map(DEPT_TO_GROUP)

    work = work.dropna(subset=["Grade distribution", "Group"])
    work = work.drop_duplicates(subset=["Term", "Course", "Professor", "Grade distribution"])

    work["Term Date"] = work["Term"].apply(term_to_date)
    work = work.dropna(subset=["Term Date"])  # quarter-only

    parsed = work["Grade distribution"].apply(parse_grade_distribution)
    parsed = parsed[parsed.map(bool)]
    work = work.loc[parsed.index].copy()

    parsed_df = parsed.apply(pd.Series).fillna(0)
    parsed_df = parsed_df.apply(pd.to_numeric, errors="coerce").fillna(0)

    total_students = parsed_df.get("Total Students", pd.Series(0, index=parsed_df.index))
    observed_outcomes = parsed_df.drop(columns=["Total Students"], errors="ignore").sum(axis=1)
    quality_ratio = observed_outcomes.div(total_students.where(total_students > 0))

    min_ratio, max_ratio = QUALITY_RATIO_RANGE
    valid = total_students.gt(0) & quality_ratio.between(min_ratio, max_ratio)

    parsed_df = parsed_df.loc[valid].copy()
    work = work.loc[valid].copy()

    band_counts = pd.DataFrame(index=parsed_df.index)
    for band, grades in BAND_TO_GRADES.items():
        band_counts[band] = parsed_df.reindex(columns=grades, fill_value=0).sum(axis=1)

    records = work[["Group", "Term", "Term Date"]].join(band_counts)
    records["Class Count"] = 1
    records["Letter Total"] = records[list(BAND_TO_GRADES.keys())].sum(axis=1)
    records = records[records["Letter Total"] > 0].copy()

    group_term = (
        records
        .groupby(["Group", "Term", "Term Date"], as_index=False)[["A", "B", "C", "D/F", "Class Count", "Letter Total"]]
        .sum()
    )
    group_term = group_term.sort_values(["Group", "Term Date", "Term"]).reset_index(drop=True)

    for band in BAND_TO_GRADES:
        group_term[f"{band} %"] = (group_term[band] / group_term["Letter Total"]) * 100

    coverage = (
        group_term
        .groupby("Group", as_index=False)
        .agg(
            Quarter_Count=("Term", "nunique"),
            First_Term=("Term", "first"),
            Last_Term=("Term", "last"),
            Class_Count=("Class Count", "sum"),
            Letter_Outcomes=("Letter Total", "sum"),
        )
        .sort_values(["Quarter_Count", "Class_Count"], ascending=[False, False])
        .reset_index(drop=True)
    )

    eligible = coverage[coverage["Quarter_Count"] >= MIN_QUARTERS].head(MAX_GROUPS)
    selected_groups = eligible["Group"].tolist()

    filtered = group_term[group_term["Group"].isin(selected_groups)].copy()
    filtered = filtered.sort_values(["Group", "Term Date", "Term"]).reset_index(drop=True)

    return filtered, coverage, selected_groups


In [ ]:
group_term, coverage_table, selected_groups = prepare_group_term_data(sunset_df)

print(f"Selected groups ({len(selected_groups)}): {selected_groups}")
coverage_table[coverage_table["Group"].isin(selected_groups)]


In [ ]:

def plot_group_trends(group_term_df, groups, with_llm_markers=False, save_path=None):
    n = len(groups)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 3.6 * nrows), sharey=True)
    axes = np.array(axes).reshape(-1)

    band_colors = {
        "A %": "#2ca02c",
        "B %": "#1f77b4",
        "C %": "#ff7f0e",
        "D/F %": "#d62728",
    }

    llm_colors = {
        "GPT": "#111111",
        "Claude": "#6f4e7c",
        "Gemini": "#1e88e5",
        "Grok": "#ff1493",
    }

    for i, group in enumerate(groups):
        ax = axes[i]
        sub = group_term_df[group_term_df["Group"] == group].sort_values(["Term Date", "Term"])

        for col, color in band_colors.items():
            ax.plot(sub["Term Date"], sub[col], marker="o", linewidth=1.9, markersize=3.7, color=color, label=col)

        # COVID context so we do not over-attribute shifts to LLMs.
        ax.axvspan(COVID_START, COVID_REMOTE_END, color="#7f7f7f", alpha=0.08)
        ax.axvline(COVID_START, color="#7f7f7f", linestyle=":", linewidth=1.2, alpha=0.9)
        ax.axvline(COVID_REMOTE_END, color="#7f7f7f", linestyle=":", linewidth=1.0, alpha=0.75)

        if with_llm_markers:
            for provider, release_date in LLM_MARKERS.items():
                ax.axvline(release_date, color=llm_colors[provider], linestyle="--", linewidth=1.2, alpha=0.7)

        ax.set_title(group)
        ax.set_ylim(0, 100)
        ax.grid(alpha=0.18)
        ax.tick_params(axis="x", rotation=35)

    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    band_handles = [plt.Line2D([0], [0], color=c, lw=2) for c in band_colors.values()]
    band_labels = list(band_colors.keys())
    covid_handle = plt.Line2D([0], [0], color="#7f7f7f", lw=1.5, linestyle=":")

    if with_llm_markers:
        marker_handles = [plt.Line2D([0], [0], color=c, lw=1.5, linestyle="--") for c in llm_colors.values()]
        marker_labels = [f"{k} release" for k in llm_colors.keys()]
        fig.legend(
            band_handles + marker_handles + [covid_handle],
            band_labels + marker_labels + ["COVID period marker"],
            loc="upper center",
            ncol=5,
            bbox_to_anchor=(0.5, 1.02),
        )
        fig.suptitle("Merged Departments: Grade-Band Change vs Time (with LLM + COVID markers)", y=1.08)
    else:
        fig.legend(
            band_handles + [covid_handle],
            band_labels + ["COVID period marker"],
            loc="upper center",
            ncol=5,
            bbox_to_anchor=(0.5, 1.02),
        )
        fig.suptitle("Merged Departments: Grade-Band Change vs Time (with COVID marker)", y=1.06)

    fig.supxlabel("Term Date")
    fig.supylabel("Share of Letter Grades (%)")
    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches="tight")

    plt.show()


In [ ]:
# Plot set 1: grade-band changes vs time (no LLM overlay)
out1 = Path("figures") / "merged_groups_grade_change_vs_time.png"
plot_group_trends(group_term, selected_groups, with_llm_markers=False, save_path=out1)
out1


In [ ]:
# Plot set 2: same grade-band trends + all four LLM markers (GPT/Claude/Gemini/Grok)
out2 = Path("figures") / "merged_groups_grade_change_with_llm_markers.png"
plot_group_trends(group_term, selected_groups, with_llm_markers=True, save_path=out2)
out2


In [ ]:
def load_provider_releases(provider, csv_path):
    updates = pd.read_csv(csv_path)
    if "Time" not in updates.columns or "Model" not in updates.columns:
        return pd.DataFrame(columns=["Model", "Release Date"])

    releases = updates[["Model", "Time"]].copy()
    releases["Release Date"] = pd.to_datetime(releases["Time"], errors="coerce", format="mixed")
    releases = releases.dropna(subset=["Release Date"]).sort_values("Release Date").reset_index(drop=True)
    releases["Provider"] = provider
    return releases[["Provider", "Model", "Release Date"]]


provider_releases = {}
for provider, path in provider_release_paths.items():
    provider_releases[provider] = load_provider_releases(provider, path)

pd.DataFrame({
    "Provider": list(provider_releases.keys()),
    "Release_Count": [len(df) for df in provider_releases.values()],
})


In [ ]:

def plot_group_trends_for_provider(group_term_df, groups, provider, releases_df, save_path=None):
    n = len(groups)
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 3.8 * nrows), sharey=True)
    axes = np.array(axes).reshape(-1)

    band_colors = {
        "A %": "#2ca02c",
        "B %": "#1f77b4",
        "C %": "#ff7f0e",
        "D/F %": "#d62728",
    }
    provider_color = {
        "GPT": "#111111",
        "Claude": "#6f4e7c",
        "Gemini": "#1e88e5",
        "Grok": "#ff1493",
    }.get(provider, "#333333")

    min_date = group_term_df["Term Date"].min()
    max_date = group_term_df["Term Date"].max()
    rel = releases_df[(releases_df["Release Date"] >= min_date - pd.Timedelta(days=45)) & (releases_df["Release Date"] <= max_date + pd.Timedelta(days=45))].copy()

    for i, group in enumerate(groups):
        ax = axes[i]
        sub = group_term_df[group_term_df["Group"] == group].sort_values(["Term Date", "Term"])

        for col, color in band_colors.items():
            ax.plot(sub["Term Date"], sub[col], marker="o", linewidth=1.9, markersize=3.6, color=color, label=col)

        # COVID context so we separate pandemic-era effects from LLM-era effects.
        ax.axvspan(COVID_START, COVID_REMOTE_END, color="#7f7f7f", alpha=0.08)
        ax.axvline(COVID_START, color="#7f7f7f", linestyle=":", linewidth=1.2, alpha=0.9)
        ax.axvline(COVID_REMOTE_END, color="#7f7f7f", linestyle=":", linewidth=1.0, alpha=0.75)

        for _, row in rel.iterrows():
            ax.axvline(row["Release Date"], color=provider_color, linestyle="--", linewidth=1.1, alpha=0.5)

        ax.set_title(group)
        ax.set_ylim(0, 100)
        ax.grid(alpha=0.18)
        ax.tick_params(axis="x", rotation=35)

    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    # Model labels on top subplot so each version is visibly marked without cluttering all panels.
    if len(rel) and n > 0:
        top_ax = axes[0]
        y_top = top_ax.get_ylim()[1]
        for _, row in rel.iterrows():
            top_ax.text(row["Release Date"], y_top * 0.98, str(row["Model"]), rotation=90, va="top", ha="center", color=provider_color, fontsize=6.8, alpha=0.8)

    band_handles = [plt.Line2D([0], [0], color=c, lw=2) for c in band_colors.values()]
    marker_handle = plt.Line2D([0], [0], color=provider_color, lw=1.5, linestyle="--")
    covid_handle = plt.Line2D([0], [0], color="#7f7f7f", lw=1.5, linestyle=":")
    legend_labels = list(band_colors.keys()) + [f"{provider} version release", "COVID period marker"]
    fig.legend(band_handles + [marker_handle, covid_handle], legend_labels, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.02))

    fig.suptitle(f"Merged Departments: Grade-Band Change vs Time with {provider} Releases + COVID marker", y=1.06)
    fig.supxlabel("Term Date")
    fig.supylabel("Share of Letter Grades (%)")
    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches="tight")

    plt.show()


In [ ]:
# Plot set 3: one plot per provider with each version release marked
provider_plot_paths = {}
for provider in ["GPT", "Claude", "Gemini", "Grok"]:
    save_path = Path("figures") / f"merged_groups_grade_change_{provider.lower()}_versions.png"
    plot_group_trends_for_provider(
        group_term_df=group_term,
        groups=selected_groups,
        provider=provider,
        releases_df=provider_releases[provider],
        save_path=save_path,
    )
    provider_plot_paths[provider] = save_path

provider_plot_paths
